# ACTIVE-TB-S: Gold-Free Testbench Synthesis Pilot

This phase consumes only sanitized specifications and model-generated oracle candidates. It freezes four ablation variants before any golden RTL, official testbench, or benchmark mutant is available.


In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys, zipfile
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
markers=list(INPUT.rglob('active_tb_s_synthesis_pilot_v1/SYNTHESIS_BOUNDARY.json'))
if len(markers)==1: SOURCE=markers[0].parent
else:
    archives=list(INPUT.rglob('active_tb_s_synthesis_pilot_v1.zip')); assert len(archives)==1, archives
    dst=WORK/'attached_synthesis_source'
    with zipfile.ZipFile(archives[0]) as zf: zf.extractall(dst)
    SOURCE=dst/'active_tb_s_synthesis_pilot_v1'
PROJECT=WORK/'active_tb_s_synthesis_project'
if PROJECT.exists(): shutil.rmtree(PROJECT)
shutil.copytree(SOURCE/'project', PROJECT)
OUT=WORK/'active_tb_s_frozen_pilot_v1'; OUT.mkdir(exist_ok=True)
print('Gold-free synthesis source:', SOURCE)


In [ ]:
boundary=json.loads((SOURCE/'SYNTHESIS_BOUNDARY.json').read_text())
assert boundary['golden_artifacts_in_dataset'] is False
assert not list(SOURCE.rglob('*mutant*'))
spec_rows=[json.loads(x) for x in (SOURCE/'spec_tasks_pilot.jsonl').read_text().splitlines() if x.strip()]
assert len(spec_rows)==20 and all(set(r)=={'task_id','description','header'} for r in spec_rows)
print('GOLD_FREE_SYNTHESIS_AUDIT_PASS:', len(spec_rows), 'tasks')


In [ ]:
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','iverilog','yosys','z3'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','vcdvcd'],check=True)
subprocess.run([sys.executable,str(PROJECT/'typed_temporal_smoke_test.py')],check=True,cwd=PROJECT)
cmd=[sys.executable,str(PROJECT/'run_spec_only_synthesis.py'),
 '--tasks',str(SOURCE/'spec_tasks_pilot.jsonl'),
 '--candidates',*[str(SOURCE/name) for name in boundary['candidate_sources']],
 '--output',str(OUT),'--cycle-budget','32','--scenario-pool-size','96',
 '--min-oracles','3','--min-consensus','0.75','--pseudo-mutants','24',
 '--max-formal-witness-calls','3','--formal-timeout','10','--resume']
subprocess.run(cmd,check=True,cwd=PROJECT)


In [ ]:
import pandas as pd
summary=pd.read_csv(OUT/'synthesis_summary.csv')
assert len(summary)==20*len(boundary['methods'])
report=summary.groupby('method',as_index=False).agg(tasks=('task_id','count'),supported=('supported','sum'),mean_scenarios=('suite_scenarios','mean'))
print(report.to_string(index=False))
manifest=json.loads((OUT/'FROZEN_BEFORE_EVALUATION.json').read_text())
assert manifest['generation_input_fields']==['task_id','description','header']
archive=WORK/'active_tb_s_frozen_pilot_v1.zip'
shutil.make_archive(str(archive.with_suffix('')),'zip',OUT)
print('Frozen checkpoint:',archive,archive.stat().st_size,'bytes')
